In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Cross_System_Monitoring")
    # Delta Lake Configurations
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.file.impl", "org.apache.hadoop.fs.local.LocalFs")
    # Network configurations to prevent Py4J Java timeouts
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

load delta tables

In [2]:
crm = spark.read.format("delta").load("../silver/crm")

billing = spark.read.format("delta").load("../silver/billing")

join records

In [ ]:
missing_in_billing = crm.join( billing,"customer_id","left_anti")

missing_in_billing.show()

+-----------+--------------+--------------------+-----------+-------------+
|customer_id|          name|               email|signup_date|         city|
+-----------+--------------+--------------------+-----------+-------------+
|  CRM000002|   Aarav Joshi|aarav.joshi375@ho...| 2022-06-28|        Surat|
|  CRM000005|    Amit Sinha|amit.sinha514@yah...| 2022-09-26|    Hyderabad|
|  CRM000009|    Reyan Shah|reyan.shah927@yah...| 2023-01-05|       Jaipur|
|  CRM000012|   Reyan Sinha|reyan.sinha219@ho...| 2023-11-20|    Bangalore|
|  CRM000014|  Priya Mishra|priya.mishra766@g...| 2022-10-04|      Kolkata|
|  CRM000015| Ishaan Sharma|ishaan.sharma115@...| 2022-03-27|    Bangalore|
|  CRM000021|  Vihaan Patel|vihaan.patel950@r...| 2023-08-14|    Hyderabad|
|  CRM000025|   Sneha Verma|sneha.verma924@ya...| 2023-07-24|      Lucknow|
|  CRM000028|  Aadhya Patel|aadhya.patel636@g...| 2023-07-24|    Bangalore|
|  CRM000030|  Kavya Tiwari|kavya.tiwari820@r...| 2023-06-19|        Thane|
|  CRM000031

Join records

In [ ]:
missing_in_crm = billing.join( crm, "customer_id","left_anti")

missing_in_crm.show()

+-----------+--------------+-------+----------------+---------+
|customer_id|transaction_id| amount|transaction_date|   status|
+-----------+--------------+-------+----------------+---------+
| GHOST00365|    TXN0000019|  95.39|      2022-07-14|completed|
| GHOST00750|    TXN0000023|  689.1|      2023-12-02|completed|
| GHOST01074|    TXN0000031| 240.42|      2024-06-23|completed|
| GHOST00109|    TXN0000049| 751.23|      2022-05-02|completed|
| GHOST00016|    TXN0000054| 683.04|      2022-09-29|   failed|
| GHOST00204|    TXN0000062|1394.08|      2023-10-12|completed|
| GHOST00977|    TXN0000065| 357.69|      2022-03-15|completed|
| GHOST01081|    TXN0000066|1487.76|      2023-04-26|completed|
| GHOST00119|    TXN0000068| 907.32|      2024-02-11|completed|
| GHOST00635|    TXN0000114| 156.88|      2022-12-08|completed|
| GHOST00663|    TXN0000130|  87.22|      2023-07-28|completed|
| GHOST00552|    TXN0000138|  133.6|      2023-09-14|completed|
| GHOST00561|    TXN0000140| 182.54|    

In [5]:
from pyspark.sql.functions import count

crm_duplicates = crm.groupBy("customer_id") \
    .agg(count("*").alias("count")) \
    .filter("count>1")

crm_duplicates.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



join records

In [ ]:
customer_match = crm.join( billing, "customer_id", "inner")

customer_match.show()

+-----------+-------------+--------------------+-----------+-------------+--------------+-------+----------------+---------+
|customer_id|         name|               email|signup_date|         city|transaction_id| amount|transaction_date|   status|
+-----------+-------------+--------------------+-----------+-------------+--------------+-------+----------------+---------+
|  CRM000001|  Rahul Singh|rahul.singh56@out...| 2024-06-20|        Delhi|    TXN0008709| 239.43|      2023-02-19|completed|
|  CRM000001|  Rahul Singh|rahul.singh56@out...| 2024-06-20|        Delhi|    TXN0004517|  61.76|      2023-01-25|completed|
|  CRM000003|   Rohan Shah|rohan.shah787@yah...| 2024-02-19|Visakhapatnam|    TXN0006487|2889.28|      2023-10-13|completed|
|  CRM000003|   Rohan Shah|rohan.shah787@yah...| 2024-02-19|Visakhapatnam|    TXN0001462|1609.76|      2023-07-08|completed|
|  CRM000004|  Arjun Singh|arjun.singh555@ou...| 2024-02-20|       Mumbai|    TXN0008990|  87.08|      2022-02-25|completed|


Saving data into delta table (gold)

In [7]:
missing_in_billing.write.format("delta").mode("overwrite").save("../gold/missing_records")

crm_duplicates.write.format("delta").mode("overwrite").save("../gold/duplicates")